### Complaint Agent — Eval Dataset Setup

Builds a curated evaluation dataset for the Complaint Agent and registers
it as a UC-managed MLflow dataset at
`{CATALOG}.evaluations.complaint_agent_eval_dataset`.

**Why a separate task:** matches the pattern in
`demos/operational-dashboard-demo/evaluation.ipynb` for the Operational
Supervisor — dataset creation is a one-time, governed, unconditional step;
evaluation is the ongoing, opt-in step. Splitting them means:

- The dataset is visible in Catalog Explorer after every deploy, even when
  `SKIP_EVAL=true` (the default for `Complaint_Evaluation`).
- Re-running the eval task doesn't rebuild the dataset — same scenarios
  every time → reproducible eval runs.
- A future MemAlign / `optimize_prompts()` job can consume the same UC
  dataset without re-deriving it.

This task runs after `Complaint_Agent` (so the streaming pipeline has had
time to land delivered orders into `{CATALOG}.lakeflow.all_events`) and
before `Complaint_Evaluation` (which reads from this dataset).

In [ ]:
%pip install -U -qqqq mlflow-skinny[databricks]
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
print(f"CATALOG = {CATALOG}")

In [ ]:
import random

all_order_ids = [
    row["order_id"]
    for row in spark.sql(
        f"""
        SELECT DISTINCT order_id
        FROM {CATALOG}.lakeflow.all_events
        WHERE event_type='delivered'
        LIMIT 50
        """
    ).collect()
]

complaint_scenarios = []

for oid in all_order_ids[:8]:
    complaint_scenarios.extend([
        f"My order took forever to arrive! Order ID: {oid}",
        f"Been waiting 2 hours, this is ridiculous. Order {oid}",
        f"Order {oid} arrived late and cold",
        f"Delivery was slower than usual for order {oid}",
    ])

for oid in all_order_ids[8:12]:
    complaint_scenarios.extend([
        f"My falafel was completely soggy and inedible. Order: {oid}",
        f"The food was cold when it arrived, very disappointing. Order: {oid}",
        f"Everything tasted bad. Order {oid}",
        f"The gyro meat was overcooked and dry, very disappointing. Order: {oid}",
    ])

for oid in all_order_ids[12:16]:
    complaint_scenarios.extend([
        f"My entire falafel bowl was missing from the order! Order: {oid}",
        f"No drinks or sides in my order {oid}",
        f"Missing my gyro from order {oid}",
        f"You forgot half my items. {oid}",
    ])

for oid in all_order_ids[16:18]:
    complaint_scenarios.extend([
        f"Where are my chicken wings?! Order {oid}",
        f"Missing my pizza from order {oid}",
    ])

for oid in all_order_ids[18:20]:
    complaint_scenarios.extend([
        f"Your driver was extremely rude to me. Order: {oid}",
        f"Driver left my food at wrong address. Order: {oid}",
    ])

for oid in all_order_ids[20:22]:
    complaint_scenarios.extend([
        f"Order {oid} was late AND missing items AND cold!",
        f"Late delivery, rude driver, and food quality was poor. Order: {oid}",
    ])

for oid in all_order_ids[22:24]:
    complaint_scenarios.extend([
        f"I'm calling my lawyer about this terrible service! Order: {oid}",
        f"This food made me sick, possible food poisoning. Order: {oid}",
        f"Found a piece of plastic in my food! Order {oid} - this is dangerous!",
    ])

complaint_scenarios.extend([
    "My order was really late and the food was cold!",
    "terrible service, do better",
    "Order ABC123 never arrived",
])

random.seed(42)
complaint_scenarios = random.sample(complaint_scenarios, min(12, len(complaint_scenarios)))

EVAL_DATASET = [
    {"inputs": {"input": [{"role": "user", "content": complaint}]}}
    for complaint in complaint_scenarios
]

print(f"Built {len(EVAL_DATASET)} eval scenarios spanning delays, quality, missing items, escalations, edge cases")

In [ ]:
import json as _json
import mlflow
import mlflow.genai.datasets

UC_DATASET_TABLE = f"{CATALOG}.evaluations.complaint_agent_eval_dataset"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.evaluations")

# Drop + recreate to avoid Arrow schema mismatches if a prior run wrote a
# different shape (mirrors evaluation.ipynb for the supervisor).
spark.sql(f"DROP TABLE IF EXISTS {UC_DATASET_TABLE}")

eval_dataset = mlflow.genai.datasets.create_dataset(uc_table_name=UC_DATASET_TABLE)


def _serialize_record(rec):
    """JSON-stringify list-of-primitives fields only (Arrow + mlflow.genai.datasets
    can't reliably serialize ARRAY<STRING>). Lists of dicts (chat-format
    `inputs.input` / `inputs.messages`) MUST stay native — mlflow.genai.datasets
    validates them against the chat schema and fails with `AttributeError:
    'str' object has no attribute 'get'` if they arrive as a stringified JSON.
    Eval-side reader deserializes the primitive lists back symmetrically."""
    out = {}
    for k, v in rec.items():
        if isinstance(v, dict):
            out[k] = _serialize_record(v)
        elif isinstance(v, list) and all(isinstance(x, (str, int, float, bool)) for x in v):
            out[k] = _json.dumps(v)
        else:
            out[k] = v
    return out


eval_dataset.merge_records([_serialize_record(r) for r in EVAL_DATASET])
print(f"✅ Registered {UC_DATASET_TABLE} ({len(EVAL_DATASET)} records)")

In [ ]:
dev_experiment_name = f"/Shared/{CATALOG}_complaint_agent_dev"
mlflow.set_experiment(dev_experiment_name)

with mlflow.start_run(run_name="register_eval_dataset"):
    mlflow.log_input(
        mlflow.data.from_spark(
            spark.table(UC_DATASET_TABLE),
            table_name=UC_DATASET_TABLE,
        ),
        context="eval",
    )

print(f"✅ Linked {UC_DATASET_TABLE} to experiment {dev_experiment_name}")
print(f"   Use in evaluate(): mlflow.genai.datasets.get_dataset(uc_table_name=UC_DATASET_TABLE)")